# 02. LangChain Model API - Abstracción y Framework
Para nuestro Steam Support Bot, LangChain nos ofrece:

* Abstracción: Si cambiamos de GPT-4o a un modelo local (como Llama 3) para ahorrar créditos, el código casi no cambia.

* Cadenas: Podemos encadenar la detección de un error con la búsqueda de una solución.

* Memoria: Fundamental para que el bot recuerde que ya nos preguntó.

# 1. Preparación del Entorno (Terminal Bash)
Ejecutar esto antes de empezar con el notebook:
```bash
pip install langchain langchain-openai
```
# 2. Configuración Inicial
Importamos las bibliotecas y validamos la conexión.

In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
import os

from dotenv import load_dotenv

load_dotenv() #Agregue esta funcion porque me daba error, esto lee el archivo .env para que el bot encuentre el token. Sin esto, me daba error de conexion

# Verificación de instalación
try:
    import langchain
    print(f"✓ LangChain version: {langchain.__version__}")
except ImportError:
    print("✗ LangChain no está instalado")

# Configuración del modelo con GitHub Models
# Usamos las variables de entorno configuradas en el Notebook 01
try:
    llm = ChatOpenAI(
        base_url=os.getenv("GITHUB_BASE_URL"),
        api_key=os.getenv("GITHUB_TOKEN"),
        model="gpt-4o",
        temperature=0.7,
        max_tokens=150
    )
    print("✓ Modelo LangChain configurado correctamente para el Soporte de Steam")
    print(f"Modelo: {llm.model_name} | Temp: {llm.temperature}")
    
except Exception as e:
    print(f"✗ Error: {e}. Verifica GITHUB_BASE_URL y GITHUB_TOKEN")

✓ LangChain version: 1.2.15
✓ Modelo LangChain configurado correctamente para el Soporte de Steam
Modelo: gpt-4o | Temp: 0.7


# 3. Configuraciones Avanzadas: Soporte vs. Creatividad
Ajustamos la temperatura según la tarea. Para Steam, un error técnico requiere precisión (0.1), mientras que una recomendación de juegos permite más libertad (0.9).

In [12]:
def comparacion_configuraciones_steam():
    # Configuración para SOPORTE TÉCNICO (Determinística)
    llm_soporte = ChatOpenAI(
        base_url=os.getenv("GITHUB_BASE_URL"),
        api_key=os.getenv("GITHUB_TOKEN"),
        model="gpt-4o",
        temperature=0.1, 
        max_tokens=200
    )
    
    # Configuración para RECOMENDACIONES (Creativa)
    llm_creativo = ChatOpenAI(
        base_url=os.getenv("GITHUB_BASE_URL"),
        api_key=os.getenv("GITHUB_TOKEN"),
        model="gpt-4o",
        temperature=0.9,
        max_tokens=200
    )
    
    prompt_tecnico = "¿Cómo soluciono el error de DLL faltante en un juego?"
    prompt_creativo = "Recomiéndame un juego parecido a 'Dark Souls' pero que tenga gatos. Con buenas reseñas positivas de usuarios"
    
    print("=== PRUEBA DE INGENIERÍA: TEMPERATURE ===")
    
    # Respuesta Técnica (Consistente)
    print("\n1. Respuesta de Soporte (Temp=0.1):")
    print("-" * 30)
    print(llm_soporte.invoke([HumanMessage(content=prompt_tecnico)]).content)
    
    # Respuesta Creativa (Variada)
    print("\n2. Respuesta de Recomendación (Temp=0.9):")
    print("-" * 30)
    print(llm_creativo.invoke([HumanMessage(content=prompt_creativo)]).content)

comparacion_configuraciones_steam()

=== PRUEBA DE INGENIERÍA: TEMPERATURE ===

1. Respuesta de Soporte (Temp=0.1):
------------------------------
El error de DLL faltante en un juego es un problema común que puede ocurrir por varias razones, como archivos corruptos, dependencias faltantes o configuraciones incorrectas. Aquí tienes una guía paso a paso para solucionar este problema:

---

### **1. Identifica la DLL faltante**
El mensaje de error generalmente menciona el nombre del archivo DLL que falta, como `msvcp140.dll`, `d3dx9_43.dll`, o algo similar. Anota el nombre exacto.

---

### **2. No descargues DLLs de sitios no confiables**
Aunque puede ser tentador buscar y descargar el archivo DLL faltante desde sitios web de terceros, esto no es recomendable. Muchos de estos sitios pueden ofrecer archivos infectados con malware o versiones incorrectas que podrían causar más problemas.

---

### **3. Instala las dependencias necesarias**
La mayoría de los errores de DLL faltantes están relacionados con bibliotecas de Micro

# 4. Comparación: LangChain vs OpenAI Directo

In [2]:
from openai import OpenAI

def comparar_enfoques_steam():
    prompt = "Resume qué es el 'Steam Deck' en una oración."
    
    print("=" * 60)
    print("COMPARACIÓN TÉCNICA: CLIENTES DE IA")
    print("=" * 60)
    
    # OpenAI Directo
    print("\n1. OpenAI Directo (Control total de metadatos):")
    client = OpenAI(base_url=os.getenv("GITHUB_BASE_URL"), api_key=os.getenv("GITHUB_TOKEN"))
    resp_directa = client.chat.completions.create(model="gpt-4o", messages=[{"role": "user", "content": prompt}])
    print(f"Respuesta: {resp_directa.choices[0].message.content}")
    print(f"Tokens Totales: {resp_directa.usage.total_tokens}")
    
    # LangChain
    print("\n2. LangChain (Abstracción y Escalabilidad):")
    resp_lang = llm.invoke([HumanMessage(content=prompt)])
    print(f"Respuesta: {resp_lang.content}")
    print(f"Tipo de objeto: {type(resp_lang)}")

comparar_enfoques_steam()

COMPARACIÓN TÉCNICA: CLIENTES DE IA

1. OpenAI Directo (Control total de metadatos):
Respuesta: El Steam Deck es una consola portátil de videojuegos desarrollada por Valve que permite jugar títulos de PC desde la plataforma Steam y otras tiendas digitales con características similares a las de una PC gaming.
Tokens Totales: 57

2. LangChain (Abstracción y Escalabilidad):
Respuesta: El Steam Deck es una consola portátil desarrollada por Valve que permite jugar videojuegos de PC desde la biblioteca de Steam y otros servicios, combinando la potencia de un ordenador con la comodidad de un dispositivo portátil.
Tipo de objeto: <class 'langchain_core.messages.ai.AIMessage'>


# 5. Tipos de Mensajes: El Flujo de Conversación
Aquí es donde integramos los datos del hardware definidos en el anterior notebook usando la estructura de mensajes de LangChain.

In [14]:
# Recuperamos los datos de hardware definidos en el Notebook 01
USER_HARDWARE = {
    "device": "Asus Vivobook 14 X1404",
    "cpu": "Intel Core i5-1235U",
    "ram": "8GB DDR4",
    "os": "Windows 11"
}

def conversacion_soporte_con_contexto():
    # 1. Obtenemos los datos de hardware
    info_pc = f"{USER_HARDWARE['device']} ({USER_HARDWARE['cpu']}, {USER_HARDWARE['ram']}, {USER_HARDWARE['os']})"
    
    # Configuramos el limite de tokens para respuestas largas y precisas
    llm.max_tokens = 800  
    
    # 2. Definición de la estructura de mensajes
    messages = [
        # --- SYSTEM MESSAGE ---
        # Define la identidad, el tono y las reglas del bot. 
        # Es la "capa base" que el modelo debe respetar durante toda la sesión.
        SystemMessage(content=(
            f"Eres el soporte oficial de Steam. Conoces el hardware del usuario: {info_pc}. "
            "Tu tarea es dar consejos técnicos precisos sobre rendimiento. Sé amable pero realista "
            "con las limitaciones de una laptop con 8GB de RAM y procesador serie U."
        )),

        # --- HUMAN MESSAGE (Turno 1) ---
        # Representa la entrada inicial del usuario (la pregunta).
        HumanMessage(content="¿Puedo correr 'Cyberpunk 2077' en mi laptop?"),

        # --- AI MESSAGE (Memoria) ---
        # Representa la respuesta previa del bot. Se incluye aquí para que el modelo 
        # tenga "memoria" y no se contradiga en los turnos siguientes.
        # Aquí marcamos la pauta realista: Cyberpunk NO es jugable en este hardware.
        AIMessage(content=(
            "Siendo totalmente honesto, NO es recomendable. Cyberpunk 2077 requiere un mínimo de 12GB de RAM "
            "y una tarjeta gráfica dedicada tras su última actualización. Con tus 8GB de RAM y los gráficos "
            "integrados Iris Xe, el juego probablemente no inicie o tenga un rendimiento injugable (menos de 10 FPS)."
        )),
        
        # --- HUMAN MESSAGE (Turno 2) ---
        # Es la nueva consulta. El modelo procesará esto considerando todo lo anterior.
        HumanMessage(content="¿Y qué tal 'Cat Quest' o 'Hollow Knight'? ¿Esos me iran mejor?")
    ]
    
    try:
        # Invocamos al modelo pasando toda la lista de mensajes (el contexto completo)
        response = llm.invoke(messages)

        print("=== 🎮 CONVERSACIÓN DE SOPORTE PERSONALIZADA ===")
        print(response.content)
    except Exception as e:
        print(f"❌ Error en la comunicación con el modelo: {e}")

# Ejecutamos la función para ver la respuesta real
conversacion_soporte_con_contexto()

=== 🎮 CONVERSACIÓN DE SOPORTE PERSONALIZADA ===
¡Claro que sí! Tanto *Cat Quest* como *Hollow Knight* son juegos mucho menos exigentes y deberían funcionar muy bien en tu Asus Vivobook 14.

### **Cat Quest**
- Este es un juego ligero con gráficos en 2D estilo caricaturesco, ideal para sistemas como el tuyo.
- Deberías poder jugarlo sin problemas a 1080p con configuraciones altas y obtener una experiencia fluida (60 FPS).

### **Hollow Knight**
- También es un título bastante optimizado, con gráficos 2D que no requieren mucho procesamiento.
- Puedes jugarlo a 1080p en configuraciones altas o medias, y obtener un rendimiento sólido.

**Consejo adicional:**
Ambos juegos deberían funcionar mejor si cierras cualquier programa en segundo plano antes de jugarlos, ya que con 8GB de RAM, liberar memoria siempre ayuda. Si notas tirones ocasionales, considera disminuir la resolución o los efectos visuales en las opciones del juego. ¡Disfrútalos! 😊
